# Web scraping: Falcon launch tables

Source: https://en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches_(2010%E2%80%932019). HTML retrieved 27 October 2026 and retained with source attribution. This independent scrape includes Falcon Heavy and does not replace the historical 90-row modeling snapshot.

Executed reconstruction created for this report. These are not recovered copies of previously unshared learner notebooks.

In [1]:
from pathlib import Path
import pandas as pd, numpy as np, json
from IPython.display import display, Image, HTML
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = ROOT / 'data'
RESULTS = ROOT / 'results'
RESULTS.mkdir(exist_ok=True)


In [2]:
from bs4 import BeautifulSoup
from io import StringIO
html=(DATA/'wikipedia_launches_2010_2019.html').read_text()
soup=BeautifulSoup(html,'html.parser')
print('Page:',soup.title.get_text() if soup.title else 'Archived Wikipedia launch list')
tables=pd.read_html(StringIO(html)); launch_tables=[t for t in tables if 'Flight No.' in t.columns and 'Booster landing' in t.columns]
print('Launch tables:',len(launch_tables))

Page: List of Falcon 9 and Falcon Heavy launches (2010–2019) - Wikipedia
Launch tables: 7


In [3]:
parts=[]
for t in launch_tables:
    t=t.copy(); t.columns=['FlightNo','DateTime','Booster','LaunchSite','Payload','PayloadMass','Orbit','Customer','LaunchOutcome','BoosterLanding']; parts.append(t)
raw=pd.concat(parts,ignore_index=True)
# Exclude narrative rows introduced by table rowspans.
flights=raw[raw.FlightNo.astype(str).str.fullmatch(r'\d+(?:\[.*?\])*',na=False)].copy()
flights=flights.drop_duplicates(['FlightNo','DateTime'])
for col in flights.columns: flights[col]=flights[col].astype(str).str.replace(r'\[[^\]]*\]','',regex=True).str.strip()
flights['PayloadKg']=pd.to_numeric(flights.PayloadMass.str.extract(r'([\d,]+(?:\.\d+)?)\s*kg')[0].str.replace(',',''),errors='coerce')
print('Parsed launch records:',len(flights));display(flights.head(10));display(flights[['PayloadMass','PayloadKg']].head())
assert len(flights)>50
flights.to_csv(RESULTS/'web_scraped_launches.csv',index=False)

Parsed launch records: 154


,FlightNo,DateTime,Booster,LaunchSite,Payload,PayloadMass,Orbit,Customer,LaunchOutcome,BoosterLanding,PayloadKg
0,1,4 June 2010 18:45,F9 v1.0 B0003,"Cape Canaveral, SLC‑40",Dragon Spacecraft Qualification Unit,Unknown,LEO,SpaceX,Success,Failure (parachute),NaN
1,1,First flight of Falcon 9 v1.0. Used a boilerpl...,First flight of Falcon 9 v1.0. Used a boilerpl...,First flight of Falcon 9 v1.0. Used a boilerpl...,First flight of Falcon 9 v1.0. Used a boilerpl...,First flight of Falcon 9 v1.0. Used a boilerpl...,First flight of Falcon 9 v1.0. Used a boilerpl...,First flight of Falcon 9 v1.0. Used a boilerpl...,First flight of Falcon 9 v1.0. Used a boilerpl...,First flight of Falcon 9 v1.0. Used a boilerpl...,NaN
2,2,8 December 2010 15:43,F9 v1.0 B0004,"Cape Canaveral, SLC‑40",SpaceX COTS Demo Flight 1 (Dragon C101),Unknown,LEO,NASA (COTS),Success,Failure (parachute),NaN
3,2,"Maiden flight of SpaceX's Dragon capsule, cons...","Maiden flight of SpaceX's Dragon capsule, cons...","Maiden flight of SpaceX's Dragon capsule, cons...","Maiden flight of SpaceX's Dragon capsule, cons...","Maiden flight of SpaceX's Dragon capsule, cons...","Maiden flight of SpaceX's Dragon capsule, cons...","Maiden flight of SpaceX's Dragon capsule, cons...","Maiden flight of SpaceX's Dragon capsule, cons...","Maiden flight of SpaceX's Dragon capsule, cons...",NaN
4,3,22 May 2012 07:44,F9 v1.0 B0005,"Cape Canaveral, SLC‑40",SpaceX COTS Demo Flight 2 (Dragon C102),"525 kg (1,157 lb) (excl. Dragon mass)",LEO (ISS),NASA (COTS),Success,No attempt,525.0
5,3,The Dragon spacecraft demonstrated a series of...,The Dragon spacecraft demonstrated a series of...,The Dragon spacecraft demonstrated a series of...,The Dragon spacecraft demonstrated a series of...,The Dragon spacecraft demonstrated a series of...,The Dragon spacecraft demonstrated a series of...,The Dragon spacecraft demonstrated a series of...,The Dragon spacecraft demonstrated a series of...,The Dragon spacecraft demonstrated a series of...,NaN
6,4,8 October 2012 00:35,F9 v1.0 B0006,"Cape Canaveral, SLC‑40",SpaceX CRS-1 (Dragon C103),"4,700 kg (10,400 lb) (excl. Dragon mass)",LEO (ISS),NASA (CRS),Success,No attempt,4700.0
8,4,"CRS-1 was successful, but the secondary payloa...","CRS-1 was successful, but the secondary payloa...","CRS-1 was successful, but the secondary payloa...","CRS-1 was successful, but the secondary payloa...","CRS-1 was successful, but the secondary payloa...","CRS-1 was successful, but the secondary payloa...","CRS-1 was successful, but the secondary payloa...","CRS-1 was successful, but the secondary payloa...","CRS-1 was successful, but the secondary payloa...",NaN
9,5,1 March 2013 15:10,F9 v1.0 B0007,"Cape Canaveral, SLC‑40",SpaceX CRS-2 (Dragon C104),"4,877 kg (10,752 lb) (excl. Dragon mass)",LEO (ISS),NASA (CRS),Success,No attempt,4877.0
10,5,Last launch of the original Falcon 9 v1.0 laun...,Last launch of the original Falcon 9 v1.0 laun...,Last launch of the original Falcon 9 v1.0 laun...,Last launch of the original Falcon 9 v1.0 laun...,Last launch of the original Falcon 9 v1.0 laun...,Last launch of the original Falcon 9 v1.0 laun...,Last launch of the original Falcon 9 v1.0 laun...,Last launch of the original Falcon 9 v1.0 laun...,Last launch of the original Falcon 9 v1.0 laun...,NaN


,PayloadMass,PayloadKg
0,Unknown,NaN
1,First flight of Falcon 9 v1.0. Used a boilerpl...,NaN
2,Unknown,NaN
3,"Maiden flight of SpaceX's Dragon capsule, cons...",NaN
4,"525 kg (1,157 lb) (excl. Dragon mass)",525.0
